In [6]:
# generate ECG from MIT-BIH record
import numpy as np
import neurokit2 as nk
import wfdb

# path to folder (NOT zip)
data_path = "mit-bih-arrhythmia-database-1.0.0"   # make sure this folder exists
record_name = "100"   # try 100, 101, 102, etc.

# read ECG record
record = wfdb.rdrecord(f"{data_path}/{record_name}")

# extract ECG signal (channel 0 = MLII lead)
ecg = record.p_signal[:, 0]

# get correct sampling rate (MIT-BIH = 360 Hz)
fs = record.fs
print("Sampling rate:", fs)
print("ECG length:", len(ecg))

# clean ECG
clean = nk.ecg_clean(ecg, sampling_rate=fs)

# detect R-peaks
_, info = nk.ecg_peaks(clean, sampling_rate=fs)
rpeaks = info["ECG_R_Peaks"]

# segment beats
window = 150
beats = []

for r in rpeaks:
    if r - window > 0 and r + window < len(clean):
        beats.append(clean[r - window:r + window])

beats = np.array(beats)
beats = beats[:, np.newaxis, :]

Sampling rate: 360
ECG length: 650000


In [10]:
# CNN + LSTM for MIT-BIH ECG with real labels and 80/20 train-test split
# plus evaluation metrics:
# - F1 score per class
# - AUC-ROC
# - Sensitivity (recall)
# - Specificity
# - Confusion matrix

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import neurokit2 as nk
import wfdb

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    recall_score,
    confusion_matrix
)

# --------------------------------------------------
# 1. Read real MIT-BIH ECG data
# --------------------------------------------------
data_path = "mit-bih-arrhythmia-database-1.0.0"
record_name = "100"   # try 100, 101, 102, etc.

record = wfdb.rdrecord(f"{data_path}/{record_name}")
annotation = wfdb.rdann(f"{data_path}/{record_name}", "atr")

# ECG signal: use channel 0
ecg = record.p_signal[:, 0]
fs = record.fs

print("Sampling rate:", fs)
print("ECG length:", len(ecg))

# --------------------------------------------------
# 2. Clean ECG and detect R-peaks
# --------------------------------------------------
clean = nk.ecg_clean(ecg, sampling_rate=fs)
_, info = nk.ecg_peaks(clean, sampling_rate=fs)
rpeaks = info["ECG_R_Peaks"]

print("Detected R-peaks:", len(rpeaks))

# --------------------------------------------------
# 3. Read MIT-BIH annotation samples and symbols
# --------------------------------------------------
ann_samples = np.array(annotation.sample)
ann_symbols = np.array(annotation.symbol)

print("Annotated beats:", len(ann_samples))
print("Unique annotation symbols:", np.unique(ann_symbols))

# --------------------------------------------------
# 4. Segment beats and assign real labels
#    0 = normal ("N")
#    1 = abnormal (everything else)
# --------------------------------------------------
window = 150
beats = []
labels_list = []

for r in rpeaks:
    if r - window > 0 and r + window < len(clean):
        beat = clean[r - window:r + window]

        # find nearest annotation to this detected R-peak
        nearest_idx = np.argmin(np.abs(ann_samples - r))
        nearest_symbol = ann_symbols[nearest_idx]

        # only keep beat if annotation is close enough
        if abs(ann_samples[nearest_idx] - r) < 50:
            beats.append(beat)

            if nearest_symbol == "N":
                labels_list.append(0)
            else:
                labels_list.append(1)

beats = np.array(beats)
beats = beats[:, np.newaxis, :]   # shape: (num_beats, 1, 300)

labels = torch.tensor(labels_list, dtype=torch.long)

print("Beats shape:", beats.shape)
print("Labels shape:", labels.shape)
print("Normal beats:", int((labels == 0).sum()))
print("Abnormal beats:", int((labels == 1).sum()))

# --------------------------------------------------
# 5. CNN model
# --------------------------------------------------
class ECG_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(32 * 72, 64)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)

        x = torch.relu(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

# quick CNN feature check
cnn = ECG_CNN()
x = torch.tensor(beats, dtype=torch.float32)
output = cnn(x)
print("CNN output shape:", output.shape)

# --------------------------------------------------
# 6. CNN + LSTM model
# --------------------------------------------------
class ECG_Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = ECG_CNN()
        self.lstm = nn.LSTM(input_size=64, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 2)   # 2 classes: normal / abnormal

    def forward(self, x):
        batch, seq, ch, length = x.shape

        x = x.view(batch * seq, ch, length)
        x = self.cnn(x)

        x = x.view(batch, seq, -1)

        lstm_out, _ = self.lstm(x)

        x = lstm_out[:, -1, :]
        x = self.fc(x)

        return x

# --------------------------------------------------
# 7. Loss and optimizer
# --------------------------------------------------
model = ECG_Model()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --------------------------------------------------
# 8. Make sequences from beats
# --------------------------------------------------
seq_len = 10
num_beats = beats.shape[0]
usable = (num_beats // seq_len) * seq_len

beats_tensor = torch.tensor(beats[:usable], dtype=torch.float32)
beats_tensor = beats_tensor.view(-1, seq_len, 1, beats.shape[2])

labels_seq = labels[:usable]
labels_seq = labels_seq.view(-1, seq_len)[:, -1]   # label = last beat in each sequence

print("Sequence tensor shape:", beats_tensor.shape)
print("Sequence labels shape:", labels_seq.shape)

# --------------------------------------------------
# 9. 80/20 train-test split
# --------------------------------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    beats_tensor,
    labels_seq,
    test_size=0.2,
    random_state=42,
    stratify=labels_seq
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Train normal:", int((y_train == 0).sum()))
print("Train abnormal:", int((y_train == 1).sum()))
print("Test normal:", int((y_test == 0).sum()))
print("Test abnormal:", int((y_test == 1).sum()))

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

# --------------------------------------------------
# 10. One training step on training set
# --------------------------------------------------
model.train()

optimizer.zero_grad()
train_outputs = model(X_train)
train_loss = criterion(train_outputs, y_train)
train_loss.backward()
optimizer.step()

print("Training loss:", train_loss.item())

# --------------------------------------------------
# 11. Evaluate on test set
# --------------------------------------------------
model.eval()

with torch.no_grad():
    test_outputs = model(X_test)
    test_loss = criterion(test_outputs, y_test)

    # probabilities for class 1 (abnormal)
    probs = torch.softmax(test_outputs, dim=1)[:, 1]

    # predicted classes
    preds = torch.argmax(test_outputs, dim=1)

# convert to numpy for sklearn
y_true = y_test.cpu().numpy()
y_pred = preds.cpu().numpy()
y_prob = probs.cpu().numpy()

# --------------------------------------------------
# 12. Metrics
# --------------------------------------------------

# confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# F1 score per class
f1_per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1])

# AUC-ROC
# only valid if both classes appear in test set
if len(np.unique(y_true)) == 2:
    auc_roc = roc_auc_score(y_true, y_prob)
else:
    auc_roc = None

# Sensitivity = recall for positive class (abnormal = 1)
sensitivity = recall_score(y_true, y_pred, pos_label=1, zero_division=0)

# Specificity = recall for negative class (normal = 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Accuracy
accuracy = (y_pred == y_true).mean()

# --------------------------------------------------
# 13. Print results
# --------------------------------------------------
print("\n--- Test Results ---")
print("Test loss:", test_loss.item())
print("Accuracy:", accuracy)

print("\nF1 score per class:")
print("Class 0 (Normal):   ", f1_per_class[0])
print("Class 1 (Abnormal): ", f1_per_class[1])

if auc_roc is not None:
    print("\nAUC-ROC:", auc_roc)
else:
    print("\nAUC-ROC: cannot be computed because only one class is present in y_true")

print("\nSensitivity (Recall for Abnormal / Class 1):", sensitivity)
print("Specificity (Recall for Normal / Class 0):", specificity)

print("\nConfusion Matrix:")
print(cm)
print("\nTN =", tn, "FP =", fp, "FN =", fn, "TP =", tp)

Sampling rate: 360
ECG length: 650000
Detected R-peaks: 2270
Annotated beats: 2274
Unique annotation symbols: ['+' 'A' 'N' 'V']
Beats shape: (2270, 1, 300)
Labels shape: torch.Size([2270])
Normal beats: 2237
Abnormal beats: 33
CNN output shape: torch.Size([2270, 64])
Sequence tensor shape: torch.Size([227, 10, 1, 300])
Sequence labels shape: torch.Size([227])
Train size: 181
Test size: 46
Train normal: 176
Train abnormal: 5
Test normal: 45
Test abnormal: 1
Train size: 181
Test size: 46
Training loss: 0.6114272475242615

--- Test Results ---
Test loss: 0.48979452252388
Accuracy: 0.9782608695652174

F1 score per class:
Class 0 (Normal):    0.989010989010989
Class 1 (Abnormal):  0.0

AUC-ROC: 0.1777777777777778

Sensitivity (Recall for Abnormal / Class 1): 0.0
Specificity (Recall for Normal / Class 0): 1.0

Confusion Matrix:
[[45  0]
 [ 1  0]]

TN = 45 FP = 0 FN = 1 TP = 0
